In [0]:
create table if not exists identifier(:catalog || ".gold.dim_store") as
select * except(file_path,file_name,ingestion_ts) from identifier(:catalog || ".silver.stores_silver")

In [0]:
create table identifier(:catalog || ".gold.dim_product") as
select * except(tags, last_changed_snapshot_date) from identifier(:catalog || ".silver.products_scd")

In [0]:
select * from identifier(:catalog || ".silver.customers_scd")

In [0]:
create table if not exists identifier(:catalog || ".gold.dim_customer") as
with source as (
  select * from identifier(:catalog || ".silver.customers_scd")
)
select
  row_number() over (order by (select null)) as customer_sk,
  *
from source

In [0]:
create table if not exists identifier(:catalog || ".gold.dim_date") as
select
  int(date_format(dt, 'yyyyMMdd')) as date_key,
  dt as calendar_date,
  date_format(dt, 'EEEE') as day_name,
  weekofyear(dt) as iso_week,
  dayofweek(dt) in (1, 7) as is_weekend
from (
  select explode(sequence(
    date '2026-03-01',
    date '2026-03-31',
    interval 1 day
  )) as dt
)

In [0]:
create table identifier(:catalog || ".gold.fact_orders") as
select o.order_id,
date(o.order_ts) as order_date,
  c.customer_sk,
  o.customer_id,
  o.store_id,
  o.order_status,
  o.payment_method,
  sum(oi.quantity) as units,
  sum(oi.gross_amount) as gross_amount,
  sum(oi.discount_amount) as discount_amount,
  sum(oi.net_amount) as net_item_amount,
  o.delivery_fee,
  (sum(oi.net_amount) + o.delivery_fee) AS order_total,
  o.promised_mins,
  o.actual_mins,
  case
        when o.order_status = 'delivered' then
            case when o.actual_mins <= o.promised_mins then true else false end
        else null
    end as is_on_time
from fk_1361.silver.orders_silver o
inner join fk_1361.silver.order_items_silver oi
  on o.order_id = oi.order_id
inner join fk_1361.gold.dim_customer c
on o.customer_id = c.customer_id
group by
    o.order_id,
    o.order_ts,
    c.customer_sk,
    o.customer_id,
    o.store_id,
    o.order_status,
    o.payment_method,
    o.delivery_fee,
    o.promised_mins,
    o.actual_mins

In [0]:
select o.order_date,o.store_id, s.city, count(o.order_id), 

In [0]:
create table fk_1361.gold.agg_daily_store_sales as
with date_range as (
  select min(order_date) as min_date, max(order_date) as max_date
  from identifier(:catalog || ".gold.fact_orders")
),
all_dates as (
  select explode(sequence(min_date, max_date, interval 1 day)) as order_date
  from date_range
),
store_days as (
  select d.order_date, s.store_id, s.city
  from all_dates d
  cross join identifier(:catalog || ".gold.dim_store") s
),
order_agg as (
  select
    order_date,
    store_id,
    count(case when order_status = 'DELIVERED' then order_id end) as delivered_orders,
    count(case when order_status = 'CANCELLED' then order_id end) as cancelled_orders,
    sum(units) as units_sold,
    sum(gross_amount) as gross_amount,
    sum(discount_amount) as discount_amount,
    sum(net_item_amount) as net_item_revenue,
    sum(delivery_fee) as delivery_fee_revenue,
    count(case when is_on_time = true then 1 end) as on_time_orders
  from identifier(:catalog || ".gold.fact_orders")
  group by order_date, store_id
)
select
  sd.order_date,
  sd.store_id,
  sd.city,
  coalesce(oa.delivered_orders, 0) as delivered_orders,
  coalesce(oa.cancelled_orders, 0) as cancelled_orders,
  coalesce(oa.units_sold, 0) as units_sold,
  cast(coalesce(oa.gross_amount, 0) as decimal(12,2)) as gross_amount,
  cast(coalesce(oa.discount_amount, 0) as decimal(12,2)) as discount_amount,
  cast(coalesce(oa.net_item_revenue, 0) as decimal(12,2)) as net_item_revenue,
  cast(coalesce(oa.delivery_fee_revenue, 0) as decimal(12,2)) as delivery_fee_revenue,
  coalesce(oa.on_time_orders, 0) as on_time_orders,
  case
    when coalesce(oa.delivered_orders, 0) = 0 then null
    else cast(100.0 * coalesce(oa.on_time_orders, 0) / coalesce(oa.delivered_orders, 0) as decimal(5,2))
  end as on_time_rate_pct
from store_days sd
left join order_agg oa
  on sd.order_date = oa.order_date
  and sd.store_id = oa.store_id
order by sd.order_date, sd.store_id

In [0]:
-- create or replace view identifier(:catalog || ".gold.kpi_overall") as
with agg as (
  select
    count(case when order_status = 'DELIVERED' then order_id end) as delivered_orders,
    count(case when order_status = 'CANCELLED' then order_id end) as cancelled_orders,
    sum(net_item_amount) as net_item_revenue,
    count(case when is_on_time = true then 1 end) as on_time_orders,
    count(distinct case when order_status = 'DELIVERED' then customer_id end) as active_customers
  from identifier(:catalog || ".gold.fact_orders")
),
customer_delivered as (
  select customer_id, count(*) as delivered_count
  from identifier(:catalog || ".gold.fact_orders")
  where order_status = 'DELIVERED'
  group by customer_id
)
select
  a.delivered_orders,
  a.cancelled_orders,
  cast(a.net_item_revenue as decimal(12,2)) as net_item_revenue,
  case
    when a.delivered_orders = 0 then null
    else cast(a.net_item_revenue / a.delivered_orders as decimal(12,2))
  end as avg_order_value,
  case
    when a.delivered_orders = 0 then null
    else cast(100.0 * a.on_time_orders / a.delivered_orders as decimal(5,2))
  end as on_time_rate_pct,
  case
    when (a.delivered_orders + a.cancelled_orders) = 0 then null
    else cast(100.0 * a.cancelled_orders / (a.delivered_orders + a.cancelled_orders) as decimal(5,2))
  end as cancellation_rate_pct,
  a.active_customers,
  case
    when a.active_customers = 0 then null
    else cast(100.0 * rc.repeat_customers / a.active_customers as decimal(5,2))
  end as repeat_customer_rate_pct
from agg a
cross join (select count(*) as repeat_customers
            from customer_delivered cd
            where cd.delivered_count >= 2) rc
left join (select 1 as dummy) d
  on true
group by
  a.delivered_orders,
  a.cancelled_orders,
  a.net_item_revenue,
  a.on_time_orders,
  a.active_customers,
  rc.repeat_customers

In [0]:
-- create table if not exists identifier(:catalog || ".gold.fact_order_line") as
select
  oi.order_id,
  oi.line_number,
  int(date_format(date(o.order_ts), 'yyyyMMdd')) as order_date_key,
  c.customer_sk,
  o.store_id,
  oi.product_id,
  o.order_status,
  oi.quantity,
  oi.unit_price,
  oi.gross_amount,
  oi.discount_amount,
  oi.net_amount
from identifier(:catalog || ".silver.order_items_silver") oi
inner join identifier(:catalog || ".silver.orders_silver") o
  on oi.order_id = o.order_id
inner join identifier(:catalog || ".gold.dim_customer") c
  on o.customer_id = c.customer_id